# QC / Temporal-Expansion Statistics (Supplementary Note 2)

Summarizes the temporal gap-filling quality of the AdDSWE archive (mean per-pixel
expansion by sensor era, season, calendar month, and year) reported in **Supplementary
Note 2**.

- **Produces:** the QC statistics in Supplementary Note 2 (printed tables).
- **Inputs:** `LS_AdDSWE_monthly_summary_*.csv` (Dryad `inundation/`).

> The input CSV path is set within the notebook (to be pointed at a local Dryad copy).


In [ ]:
import os

# =============================== PATH CONFIGURATION ===============================
# Point DRYAD_ROOT at your local copy of the Dryad archive (doi:10.5061/dryad.msbcc2gct).
DRYAD_ROOT = r"."            # e.g. r"E:\Okavango\Data\For_Dryad"
OUTPUT_DIR = r"./outputs"    # local folder for derived CSVs / figures
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np

# ============================================================================
# LOAD DATA
# ============================================================================

df = pd.read_csv(os.path.join(DRYAD_ROOT, "inundation", "LS_AdDSWE_monthly_summary_01082026.csv"))

# Parse date column and extract year/month
df['date']  = pd.to_datetime(df['date'])
df['year']  = df['date'].dt.year
df['month'] = df['date'].dt.month

# Preview QC columns available
qc_cols = [c for c in df.columns if c.startswith('qc')]
print("QC columns found:", qc_cols)
print(f"Total months in archive: {len(df)}")
print(f"Date range: {df['date'].min().strftime('%b %Y')} to {df['date'].max().strftime('%b %Y')}")
print()

# ============================================================================
# 1. OVERALL ARCHIVE QUALITY
# ============================================================================

print("=" * 60)
print("1. OVERALL ARCHIVE QUALITY")
print("=" * 60)

avg = df['qc_avg_value']
print(f"  Mean qc_avg_value (all months):   {avg.mean():.3f}")
print(f"  Median qc_avg_value:              {avg.median():.3f}")
print(f"  Std dev:                          {avg.std():.3f}")
print(f"  Min / Max:                        {avg.min():.3f} / {avg.max():.3f}")
print()

pct_any   = (avg > 0).mean() * 100
pct_over1 = (avg > 1).mean() * 100
pct_over3 = (avg > 3).mean() * 100
print(f"  X: Months requiring ANY expansion (qc_avg > 0):   {pct_any:.1f}%")
print(f"  Y: Months requiring >1 month avg  (qc_avg > 1):   {pct_over1:.1f}%")
print(f"  Z: Months requiring >3 month avg  (qc_avg > 3):   {pct_over3:.1f}%")
print()

# ============================================================================
# 2. QUALITY BY SENSOR ERA
# ============================================================================

print("=" * 60)
print("2. QUALITY BY SENSOR ERA")
print("=" * 60)

def era_label(year):
    if year < 1999:
        return '1984-1998 (LS5 only)'
    elif year < 2003:
        return '1999-2002 (LS5+7)'
    elif year < 2012:
        return '2003-2012 (LS7 SLC-off / LS5)'
    elif year < 2014:
        return '2013 (LS8 only, transition)'
    else:
        return '2014-2025 (LS8/9)'

df['era'] = df['year'].apply(era_label)
era_summary = df.groupby('era')['qc_avg_value'].agg(['mean','median','std','count'])
era_summary.columns = ['Mean','Median','Std','N months']
era_summary = era_summary.sort_values('Mean', ascending=False)
print(era_summary.round(3).to_string())
print()

# ============================================================================
# 3. QUALITY BY CALENDAR MONTH (seasonal pattern)
# ============================================================================

print("=" * 60)
print("3. QUALITY BY CALENDAR MONTH (seasonal pattern)")
print("=" * 60)

month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
month_summary = df.groupby('month')['qc_avg_value'].agg(['mean','median','std','count'])
month_summary.columns = ['Mean','Median','Std','N years']
month_summary.index = [month_names[m] for m in month_summary.index]
print(month_summary.round(3).to_string())
print()

df['season'] = df['month'].apply(
    lambda m: 'Wet (Nov-Apr)' if m in [11,12,1,2,3,4] else 'Dry (May-Oct)'
)
season_summary = df.groupby('season')['qc_avg_value'].agg(['mean','median','std','count'])
season_summary.columns = ['Mean','Median','Std','N months']
print("Wet vs Dry season summary:")
print(season_summary.round(3).to_string())
print()

# ============================================================================
# 4. WORST AND BEST MONTHS
# ============================================================================

print("=" * 60)
print("4. WORST 10 AND BEST 10 MONTHS BY qc_avg_value")
print("=" * 60)

display_cols = ['date','qc_avg_value','qc_pixel_count','qc_area_km2']
print("WORST 10 months (highest qc_avg_value):")
print(df.nlargest(10, 'qc_avg_value')[display_cols].to_string(index=False))
print()
print("BEST 10 months (lowest qc_avg_value, excluding 0):")
best = df[df['qc_avg_value'] > 0].nsmallest(10, 'qc_avg_value')
print(best[display_cols].to_string(index=False))
print()

# ============================================================================
# 5. DISTRIBUTION OF qc_avg_value ACROSS FULL ARCHIVE
# ============================================================================

print("=" * 60)
print("5. DISTRIBUTION OF qc_avg_value (binned)")
print("=" * 60)

bins   = [0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.01]
labels = ['0–0.5','0.5–1.0','1.0–1.5','1.5–2.0','2.0–3.0','3.0–4.0','>4.0']
df['qc_bin'] = pd.cut(df['qc_avg_value'], bins=bins, labels=labels, right=False)
bin_counts = df['qc_bin'].value_counts().sort_index()
bin_pct    = (bin_counts / len(df) * 100).round(1)
bin_df     = pd.DataFrame({'Count': bin_counts, 'Pct (%)': bin_pct})
print(bin_df.to_string())
print()

# ============================================================================
# 6. ANNUAL MEAN qc_avg_value
# ============================================================================

print("=" * 60)
print("6. ANNUAL MEAN qc_avg_value (all years)")
print("=" * 60)

annual = df.groupby('year')['qc_avg_value'].mean().round(3)
print(annual.to_string())
print()

print("=" * 60)
print("ANALYSIS COMPLETE")
print("=" * 60)